# <font size=6><b>Lec06.워드 임베딩(Word Embedding)
* https://wikidocs.net/50739
* Word2vector : 주변 단어
* FastText
* Glove : 전체적인 등장 비율
* Elmo : 문장 전체를 다 읽고 상황에 맞는 벡터 생성

* <b>희소 표현(Sparse Representation)</b>
  - [ 0 0 0 0 1 0 0 0 0 0 0 0 ... 중략 ... 0] : 표현하고자 하는 인덱스의 값만 1, 나머지는 전부 0 -> 희소 표현(벡터의 차원이 한없이 커짐)
* <b>밀집 표현(Dense Representation)</b> 
  -  [0.2 1.8 1.1 -2.1 1.1 2.8 ... 중략 ...] : 저차원, 희소 표현과 반대되는 표현
* <font color=red><b>워드 임베딩(Word Embedding)
    - 단어를 밀집 벡터(dense vector)의 형태로 표현하는 방법

# <b>워드투벡터(Word2Vec)
* 단어 벡터 간 유사도를 수치화 할 수 있는 방법

<img src = "https://static.wikidocs.net/images/page/22660/word2vec_renew_4.PNG">

In [1]:
import re
import urllib.request
import zipfile
from lxml import etree
from nltk.tokenize import word_tokenize, sent_tokenize

In [2]:
# 데이터 다운로드
#urllib.request.urlretrieve("https://raw.githubusercontent.com/ukairia777/tensorflow-nlp-tutorial/main/09.%20Word%20Embedding/dataset/ted_en-20160408.xml", filename="lec06_ted_en-20160408.xml")

In [3]:
targetXML = open('./dataset/lec06_ted_en-20160408.xml', 'r', encoding='UTF8')
target_text = etree.parse(targetXML)

# xml 파일로부터 <content>와 </content> 사이의 내용만 가져온다.
parse_text = '\n'.join(target_text.xpath('//content/text()'))
# parse_text  ->대용량이라 프린트하지말기

In [4]:
# 정규 표현식의 sub 모듈을 통해 content 중간에 등장하는 괄호 (Audio), (Laughter) 등의 배경음 부분을 제거.
# 해당 코드는 괄호로 구성된 내용을 제거. 정규화 과정
content_text = re.sub(r'\([^)]*\)', '', parse_text)

In [5]:
# NLTK로 문장 토큰화
sent_text = sent_tokenize(content_text)
sent_text[:2]

["Here are two reasons companies fail: they only do more of the same, or they only do what's new.",
 'To me the real, real solution to quality growth is figuring out the balance between two activities: exploration and exploitation.']

In [6]:
# 각 문장에 대해서 구두점을 제거하고, 대문자를 소문자로 변환.
normalized_text = []
for string in sent_text:
     tokens = re.sub(r"[^a-z0-9]+", " ", string.lower())
     normalized_text.append(tokens)
normalized_text[:2]

['here are two reasons companies fail they only do more of the same or they only do what s new ',
 'to me the real real solution to quality growth is figuring out the balance between two activities exploration and exploitation ']

In [7]:
# 각 문장에 대해서 NLTK를 이용하여 단어 토큰화를 수행.
result = [word_tokenize(sentence) for sentence in normalized_text]

# <b>모델 생성

In [8]:
# ! pip install gensim

In [9]:
from gensim.models import Word2Vec
from gensim.models import KeyedVectors

In [10]:
model = Word2Vec(sentences=result, vector_size=100, min_count=5, sg=0) # sg = 0은 CBOW, 1은 Skip-gram

In [11]:
model_result = model.wv.most_similar("man")
print(model_result)

[('woman', 0.8392671346664429), ('guy', 0.8127705454826355), ('lady', 0.7684139013290405), ('boy', 0.7526049017906189), ('girl', 0.7272317409515381), ('soldier', 0.7126677632331848), ('gentleman', 0.7086498737335205), ('kid', 0.6733858585357666), ('poet', 0.6392357349395752), ('person', 0.6330706477165222)]


# <b>모델 저장

In [12]:
model.wv.save_word2vec_format('lec06_w2v_model') # 모델 저장
loaded_model = KeyedVectors.load_word2vec_format("lec06_w2v_model") # 모델 로드

In [13]:
model_result = loaded_model.most_similar("man")
print(model_result)

[('woman', 0.8392671346664429), ('guy', 0.8127705454826355), ('lady', 0.7684139013290405), ('boy', 0.7526049017906189), ('girl', 0.7272317409515381), ('soldier', 0.7126677632331848), ('gentleman', 0.7086498737335205), ('kid', 0.6733858585357666), ('poet', 0.6392357349395752), ('person', 0.6330706477165222)]


# ----------